# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Davionnic/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. My rule and its reason codes

**Lane (locked):** Refresh / Content Opportunity Scoring.

**Rule in plain words:** Review a page sooner when it is (a) still getting meaningful search visibility, (b) looking stale / aging, (c) sitting in a good position band where a slip would hurt, and (d) thin relative to peers. The score is a weighted mix of those four signals — no fitted weights.

**Score (same as `scripts/02_baseline_score.py`):**
```
baseline_refresh_score =
  0.40 * visibility_score          # percentile_rank(log1p(impressions_90d))
+ 0.30 * freshness_risk_score      # percentile_rank(days_since_last_update)
+ 0.25 * position_opportunity_score
+ 0.05 * depth_gap_score
```

**Reason codes the rule can output** (joined with `|`; explain *why*, not the numeric score):
| Code | Fires when |
|---|---|
| `stale_visible_page` | `days_since_last_update >= 180` and `impressions_90d >= 500` |
| `declining_with_demand` | `trend_direction == "down"` and `impressions_90d >= 100` |
| `thin_visible_page` | `0 < word_count < 1200` and `impressions_90d >= 250` |
| `page_one_decay_risk` | `0 < avg_position <= 10` and `content_age_days >= 180` |
| `low_ctr_visible_page` | `impressions_90d >= 500`, `0 < avg_position <= 20`, `ctr < 0.5` |
| `low_engagement_visible_page` | `sessions_90d >= 30` and engagement or scroll rate under 30 |
| `general_refresh_review` | none of the above |

**Suggested actions:** `expand_and_refresh` / `refresh_and_review_ctr` / `refresh` / `monitor`.

**Honest note:** `declining_with_demand` reads the label-source column for a human-readable tag. The **numeric score does not** use `trend_direction` / `trend_pct` / `is_declining_label`.


In [1]:
import os
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# walk up until we find the repo root (works from work/notebooks/ or Colab)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import normalize, percentile_rank, precision_at_k  # noqa: E402

RAW = "data/raw/content_refresh_anonymized.csv"
FEAT = "data/processed/refresh_feature_vector.csv"
assert os.path.exists(RAW), f"missing starter CSV at {os.path.abspath(RAW)}"

# Prefer the prepared feature vector (same filters/label as the reference pipeline).
# If missing, run prepare once — do not edit scripts/.
if not os.path.exists(FEAT):
    import runpy

    runpy.run_path("scripts/01_prepare_features.py", run_name="__main__")

df = pd.read_csv(FEAT)
assert "is_declining_label" in df.columns
assert (
    df["is_declining_label"]
    == (df["trend_direction"].astype(str).str.lower() == "down").astype(int)
).all()

print("Working dir:", os.getcwd())
print(f"feature rows={len(df):,}  cols={df.shape[1]}  clients={df['client_id'].nunique()}")
print("lane: Refresh / Content Opportunity Scoring")
print("label: is_declining_label = (trend_direction == 'down')")
print("metric: Precision@50 on client-holdout (RANDOM_STATE=42)")
print("data: bundled starter CSV / refresh_feature_vector (no HF warehouse)")


Working dir: /workspace/flyrank-ml-internship
feature rows=30,000  cols=52  clients=32
lane: Refresh / Content Opportunity Scoring
label: is_declining_label = (trend_direction == 'down')
metric: Precision@50 on client-holdout (RANDOM_STATE=42)
data: bundled starter CSV / refresh_feature_vector (no HF warehouse)


## 2. Build the ranked queue (writes the CSV)

Code below mirrors `scripts/02_baseline_score.py`: transparent score + reason codes + suggested action, then rank descending. Writes `work/outputs/baseline_action_score.csv` (gitignored by design) and a small metrics JSON receipt. Primary number: **Precision@50 on the same client-holdout** used by `scripts/03_train_model.py` (`RANDOM_STATE=42`, ~20% of clients).


In [2]:
def reason_codes_row(row: pd.Series) -> list[str]:
    reasons: list[str] = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if str(row["trend_direction"]).lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row["sessions_90d"] >= 30 and (
        (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
        or (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
    ):
        reasons.append("low_engagement_visible_page")
    if not reasons:
        reasons.append("general_refresh_review")
    return reasons


def suggested_action_row(row: pd.Series) -> str:
    reasons = set(str(row["reason_codes"]).split("|"))
    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"
    if "stale_visible_page" in reasons or "declining_with_demand" in reasons:
        return "refresh"
    return "monitor"


scored = df.copy()
scored["visibility_score"] = percentile_rank(np.log1p(scored["impressions_90d"]))
scored["freshness_risk_score"] = percentile_rank(scored["days_since_last_update"])
scored["position_opportunity_score"] = (
    (1 - normalize(scored["avg_position"].clip(lower=1, upper=50)))
    * scored["visibility_score"]
    * (scored["avg_position"] > 0).astype(int)
)
scored["depth_gap_score"] = (1 - percentile_rank(scored["word_count"])) * scored["visibility_score"]
scored["baseline_refresh_score"] = (
    0.40 * scored["visibility_score"]
    + 0.30 * scored["freshness_risk_score"]
    + 0.25 * scored["position_opportunity_score"]
    + 0.05 * scored["depth_gap_score"]
).clip(0, 1)
scored["reason_codes"] = scored.apply(lambda r: "|".join(reason_codes_row(r)), axis=1)
scored["suggested_action_baseline"] = scored.apply(suggested_action_row, axis=1)
scored["baseline_rank"] = (
    scored["baseline_refresh_score"].rank(method="first", ascending=False).astype(int)
)

output_columns = [
    "content_id", "client_id", "baseline_rank", "baseline_refresh_score",
    "visibility_score", "freshness_risk_score", "position_opportunity_score", "depth_gap_score",
    "reason_codes", "suggested_action_baseline", "is_declining_label",
    "impressions_90d", "clicks_90d", "sessions_90d", "avg_position", "ctr",
    "engagement_rate", "scroll_rate", "content_age_days", "days_since_last_update",
    "word_count", "trend_direction",
]
ranked = scored[output_columns].sort_values("baseline_rank")

out_dir = Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "baseline_action_score.csv"
ranked.to_csv(csv_path, index=False)

# --- client-holdout Precision@K (match scripts/03_train_model.py) ---
RANDOM_STATE = 42
client_series = scored["client_id"].fillna("unknown").astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:test_client_count].tolist())
test_mask = client_series.isin(test_clients).to_numpy()
test = scored.loc[test_mask]
train = scored.loc[~test_mask]

p10 = precision_at_k(test["is_declining_label"], test["baseline_refresh_score"], 10)
p20 = precision_at_k(test["is_declining_label"], test["baseline_refresh_score"], 20)
p50 = precision_at_k(test["is_declining_label"], test["baseline_refresh_score"], 50)
base_rate_full = float(scored["is_declining_label"].mean())
base_rate_test = float(test["is_declining_label"].mean())
full_top50 = float(ranked.head(50)["is_declining_label"].mean())

metrics = {
    "assignment": "ML-07",
    "lane": "Refresh / Content Opportunity Scoring",
    "label": "is_declining_label = (trend_direction == 'down')",
    "split": "client_holdout",
    "random_state": RANDOM_STATE,
    "test_clients": sorted(test_clients),
    "train_rows": int(len(train)),
    "test_rows": int(len(test)),
    "base_rate_full": base_rate_full,
    "base_rate_test": base_rate_test,
    "precision_at_10_holdout": p10,
    "precision_at_20_holdout": p20,
    "precision_at_50_holdout": p50,
    "declining_rate_top_50_full": full_top50,
    "score_formula": {
        "visibility_score": 0.40,
        "freshness_risk_score": 0.30,
        "position_opportunity_score": 0.25,
        "depth_gap_score": 0.05,
    },
    "output_csv": str(csv_path),
}
metrics_path = out_dir / "baseline_action_score_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2) + "\n")

print(f"Wrote ranked queue: {csv_path}")
print(
    f"rows={len(ranked):,}  score range="
    f"[{ranked['baseline_refresh_score'].min():.4f}, {ranked['baseline_refresh_score'].max():.4f}]"
)
print(f"median score={ranked['baseline_refresh_score'].median():.4f}")
print(f"Wrote metrics receipt: {metrics_path}")
print()
print("=== client-holdout evaluation (same split as scripts/03_train_model.py) ===")
print(f"test_clients={sorted(test_clients)}")
print(f"train_rows={len(train):,}  test_rows={len(test):,}")
print(f"base rate full={base_rate_full:.3f}  base rate holdout={base_rate_test:.3f}")
print(f"Precision@10 (holdout) = {p10:.3f}")
print(f"Precision@20 (holdout) = {p20:.3f}")
print(f"Precision@50 (holdout) = {p50:.3f}   << primary metric")
print(f"declining rate top-50 on FULL sample (NOT the holdout metric) = {full_top50:.3f}")


Wrote ranked queue: work/outputs/baseline_action_score.csv
rows=30,000  score range=[0.0080, 0.9412]
median score=0.4429
Wrote metrics receipt: work/outputs/baseline_action_score_metrics.json

=== client-holdout evaluation (same split as scripts/03_train_model.py) ===
test_clients=['client_0b918943df', 'client_1a6562590e', 'client_4fc82b26ae', 'client_98a3ab7c34', 'client_d4735e3a26', 'client_f74efabef1']
train_rows=27,675  test_rows=2,325
base rate full=0.542  base rate holdout=0.391
Precision@10 (holdout) = 0.200
Precision@20 (holdout) = 0.150
Precision@50 (holdout) = 0.240   << primary metric
declining rate top-50 on FULL sample (NOT the holdout metric) = 0.340


## 3. Top-20 review

Hand review of the top 20 pages from the ranked queue (pseudonym IDs only). For each band: **action**, **reason code(s)**, **confidence note**, and **what would make it wrong**.

Observed from the real run: **2/10** of the top-10 and **7/20** of the top-20 carry the proxy decline label. Every top-20 row fires `page_one_decay_risk` + `low_engagement_visible_page` — high-visibility, aged, page-1 pages. Holdout **Precision@50 = 0.240** (primary).

| Rank | Action | Reason codes (short) | Confidence | What would make it wrong |
|---:|---|---|---|---|
| 1 | refresh | declining_with_demand + page_one_decay + low_engagement | Higher — proxy down + huge impressions | Trend already reversed; engagement metric mis-scaled |
| 2–3 | monitor | page_one_decay + low_engagement | Lower — proxy **stable**, not down | Wasted review if page is healthy and still climbing |
| 4–5 | refresh_and_review_ctr | + low_ctr | Medium — CTR review may still help even if not declining | CTR threshold too strict for this niche; title/snippet already fine |
| 6 | refresh_and_review_ctr | declining + page_one + low_ctr + low_eng | Higher — proxy down + CTR flag | Same-window proxy; not proof refresh recovers |
| 7–10 | monitor / refresh_and_review_ctr | page_one + low_eng (± low_ctr) | Lower — mostly **stable** giants | Visibility alone is not decline; capacity better spent elsewhere |
| 11–20 | mixed | same family | Mixed — several proxy-down among stable giants | Rule overweights page-1 visibility vs true decline |

**Top-10 takeaway:** The top of the queue is dominated by page-1, high-impression, aged pages with low-engagement flags. Only **2 of the top 10** match the proxy decline label — so the hand rule surfaces "important pages" more than "declining pages." That is useful for a glance-first review queue, but it is a weak Precision@K baseline against `is_declining_label` (holdout Precision@50 = **0.240**, vs holdout base rate **0.391**).


In [3]:
# Top-20 table for the hand review (pseudonyms only — no URLs/titles/queries)
from collections import Counter

review_cols = [
    "baseline_rank", "content_id", "baseline_refresh_score", "is_declining_label",
    "suggested_action_baseline", "reason_codes", "impressions_90d", "avg_position",
    "ctr", "days_since_last_update", "content_age_days", "trend_direction",
]
top20 = ranked.head(20)
print("=== Top-20 of the ranked queue (full scored set; hand review) ===")
print(top20[review_cols].to_string(index=False))
rc = Counter()
for r in top20["reason_codes"]:
    for x in r.split("|"):
        rc[x] += 1
print()
print("reason-code counts in top-20:", dict(rc))
print(f"proxy positives in top-10: {int(ranked.head(10)['is_declining_label'].sum())} / 10")
print(f"proxy positives in top-20: {int(top20['is_declining_label'].sum())} / 20")


=== Top-20 of the ranked queue (full scored set; hand review) ===
 baseline_rank           content_id  baseline_refresh_score  is_declining_label suggested_action_baseline                                                                               reason_codes  impressions_90d  avg_position  ctr  days_since_last_update  content_age_days trend_direction
             1 content_9532f197bbc8                0.941189                   1                   refresh                      declining_with_demand|page_one_decay_risk|low_engagement_visible_page           309192           2.0 0.87                     104               445            down
             2 content_4d1fe5b32dc2                0.934889                   0                   monitor                                            page_one_decay_risk|low_engagement_visible_page            97999           2.5 0.52                     104               329          stable
             3 content_07f2e7a6f38a                0.934080  

## 4. Weak picks + leakage check

**Weak picks:** In the top 20, **13 / 20** are proxy-stable (`is_declining_label == 0`). Ranks 2–5, 7–11, 14, 17–19 are high-impression page-1 pages that look "important" to the visibility/age/position mix but are **not** tagged declining. That is expected for this hand rule: visibility + page-one age dominate the score; decline is only a reason-code tag, not a score term. A content team should treat those as "worth a glance," not "must refresh."

**Why they look wrong vs the label:** Precision@K is scored against `is_declining_label`. Surfacing healthy giants hurts Precision@50 even when a human might still want them monitored.

**Leakage check:**
- Numeric score inputs: `impressions_90d`, `days_since_last_update`, `avg_position`, `word_count` only — **no** `trend_direction` / `trend_pct` / `is_declining_label`.
- Reason code `declining_with_demand` *does* read `trend_direction` for explainability; it does not change the rank number.
- Product decision flags: **absent** from the starter CSV (per data contract) — nothing leaked in.
- Future windows: **not available** on the starter; label is same-window teaching proxy.
- IDs (`content_id`, `client_id`) used for join/split only, never as score features.

Claims stay **observed / directional / decision-support**. This baseline is the transparent floor a later model must beat on the same holdout.


In [4]:
# Weak picks + leakage probes
weak = ranked.head(20).loc[lambda x: x["is_declining_label"] == 0]
print("=== Weak picks in top-20 (proxy label == 0) ===")
print(f"count={len(weak)} / 20")
print(
    weak[
        [
            "baseline_rank",
            "content_id",
            "suggested_action_baseline",
            "reason_codes",
            "impressions_90d",
            "trend_direction",
        ]
    ].to_string(index=False)
)

print()
print("=== Leakage check (honest) ===")
score_inputs = {"impressions_90d", "days_since_last_update", "avg_position", "word_count"}
forbidden = {"trend_direction", "trend_pct", "is_declining_label"}
print("score components use:", ", ".join(sorted(score_inputs)))
print("score does NOT use:", ", ".join(sorted(forbidden)))
print("reason codes MAY mention declining_with_demand (uses trend_direction) — explainability only")
print("product decision flags: not present in starter CSV (nothing to strip)")
print("future windows: not on starter — same-window teaching proxy only")
overlap = score_inputs & forbidden
print("forbidden feature probe on score inputs:", "LEAK" if overlap else "OK (no label columns)")


=== Weak picks in top-20 (proxy label == 0) ===
count=13 / 20
 baseline_rank           content_id suggested_action_baseline                                                         reason_codes  impressions_90d trend_direction
             2 content_4d1fe5b32dc2                   monitor                      page_one_decay_risk|low_engagement_visible_page            97999          stable
             3 content_07f2e7a6f38a                   monitor                      page_one_decay_risk|low_engagement_visible_page           101078          stable
             4 content_e5ae436f9a16    refresh_and_review_ctr page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page           117741          stable
             5 content_3430a8b94511    refresh_and_review_ctr page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page           152617          stable
             7 content_9c195417f6ef                   monitor                      page_one_decay_risk|low_engagement_

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**ML-06 note:** `work/README.md` marks `w04_signal_audit.ipynb` as optional stretch (card retired; core folded into ML-07). This assignment completes the baseline notebook only.

*(Portal submit is out of scope for this agent run — commit + push only. Capstone untouched.)*
